**Nama : Fauzan Ahsanudin Alfikri**

**NIM : 103052300003**

**Customer_Churn_Prediction**

##Import Data

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

In [23]:
dftrain = pd.read_csv('/content/drive/MyDrive/Kompetisi/Customer Churn Prediction Challenge/train.csv')
dftest = pd.read_csv('/content/drive/MyDrive/Kompetisi/Customer Churn Prediction Challenge/test.csv')

##Prepocessing Data

In [24]:
dftrain

,ID,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Complain,Satisfaction Score,Card Type,Point Earned,Exited
0,1,Martin,727.0,Spain,Male,38.0,2,62276.99,1,Yes,Yes,59280.79,No,4,DIAMOND,757,0
1,2,Chinweuba,529.0,France,Female,29.0,8,0.00,2,Yes,NaN,19842.11,No,3,SILVER,476,0
2,3,Clapp,589.0,France,Female,50.0,4,0.00,2,No,Yes,182076.97,No,4,PLATINUM,441,0
3,4,Boni,515.0,France,Male,40.0,0,109542.29,1,Yes,Yes,166370.81,No,5,GOLD,312,0
4,5,Jamieson,528.0,Spain,Male,43.0,7,97473.87,2,Yes,Yes,159823.16,No,3,PLATINUM,654,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8025,8026,Ma,490.0,Germany,NaN,41.0,0,139659.04,1,Yes,Yes,176254.12,No,5,SILVER,312,0
8026,8027,Stevens,553.0,Spain,Male,41.0,6,144974.55,1,Yes,Yes,19344.92,No,1,PLATINUM,699,0
8027,8028,Defalco,808.0,France,Male,41.0,0,0.00,1,Yes,Yes,79888.78,No,3,GOLD,398,0
8028,8029,Docherty,706.0,France,Female,NaN,5,0.00,1,No,No,164128.41,Yes,2,GOLD,620,1


In [25]:
print("======SAMPEL DATA=====")
print(dftrain.head())
print("\n======INFO DATA=====")
print(dftrain.info())
print("\n======STATISTIK DESKRIPTIF=====")
print(dftrain.describe())
print("\n======MISSING VALUES=====")
print(dftrain.isnull().sum())
print("\n======DUPLICATE DATA=====")
print(dftrain.duplicated().sum())

======SAMPEL DATA=====
   ID    Surname  CreditScore Geography  Gender   Age  Tenure    Balance  \
0   1     Martin        727.0     Spain    Male  38.0       2   62276.99   
1   2  Chinweuba        529.0    France  Female  29.0       8       0.00   
2   3      Clapp        589.0    France  Female  50.0       4       0.00   
3   4       Boni        515.0    France    Male  40.0       0  109542.29   
4   5   Jamieson        528.0     Spain    Male  43.0       7   97473.87   

   NumOfProducts HasCrCard IsActiveMember  EstimatedSalary Complain  \
0              1       Yes            Yes         59280.79       No   
1              2       Yes            NaN         19842.11       No   
2              2        No            Yes        182076.97       No   
3              1       Yes            Yes        166370.81       No   
4              2       Yes            Yes        159823.16       No   

   Satisfaction Score Card Type  Point Earned  Exited  
0                   4   DIAMOND      

In [26]:
# Mengisi nilai yang hilang dengan modus
dftrain['Gender'].fillna(dftrain['Gender'].mode()[0], inplace=True)
dftrain['IsActiveMember'].fillna(dftrain['IsActiveMember'].mode()[0], inplace=True)

<ipython-input-26-e344873b157b>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dftrain['Gender'].fillna(dftrain['Gender'].mode()[0], inplace=True)
<ipython-input-26-e344873b157b>:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inpla

In [27]:
# Menghapus kolom non-numerik
df_numeric = dftrain.select_dtypes(include=['float64', 'int64'])

# Menggunakan KNNImputer untuk mengisi nilai yang hilang
imputer = KNNImputer(n_neighbors=2)  # Menggunakan 2 tetangga terdekat
df_imputed = imputer.fit_transform(df_numeric)

# Mengubah hasil kembali ke DataFrame
df_imputed = pd.DataFrame(df_imputed, columns=df_numeric.columns)

# Menggabungkan hasil imputasi kembali ke DataFrame asli
dftrain[df_numeric.columns] = df_imputed

In [28]:
print("\n======MISSING VALUES=====")
print(dftrain.isnull().sum())


======MISSING VALUES=====
ID                    0
Surname               0
CreditScore           0
Geography             0
Gender                0
Age                   0
Tenure                0
Balance               0
NumOfProducts         0
HasCrCard             0
IsActiveMember        0
EstimatedSalary       0
Complain              0
Satisfaction Score    0
Card Type             0
Point Earned          0
Exited                0
dtype: int64


##Train Model

In [29]:
dftrain['Gender'] = dftrain['Gender'].map({'Male': 0, 'Female': 1})
dftrain['HasCrCard'] = dftrain['HasCrCard'].map({'No': 0, 'Yes': 1})
dftrain['IsActiveMember'] = dftrain['IsActiveMember'].map({'No': 0, 'Yes': 1})
dftrain['Complain'] = dftrain['Complain'].map({'No': 0, 'Yes': 1})

# Menggunakan LabelEncoder yang sama untuk Geography dan Card Type
label_encoder_geo = LabelEncoder()
label_encoder_card = LabelEncoder()

# Fit encoder pada dftrain dan transform pada dftest
dftrain['Geography'] = label_encoder_geo.fit_transform(dftrain['Geography'])
dftrain['Card Type'] = label_encoder_card.fit_transform(dftrain['Card Type'])

# Transform dftest menggunakan encoder yang sama
dftest['Geography'] = label_encoder_geo.transform(dftest['Geography'])
dftest['Card Type'] = label_encoder_card.transform(dftest['Card Type'])

In [30]:
dftrain

,ID,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Complain,Satisfaction Score,Card Type,Point Earned,Exited
0,1.0,Martin,727.0,2,0,38.0,2.0,62276.99,1.0,1,1,59280.79,0,4.0,0,757.0,0.0
1,2.0,Chinweuba,529.0,0,1,29.0,8.0,0.00,2.0,1,1,19842.11,0,3.0,3,476.0,0.0
2,3.0,Clapp,589.0,0,1,50.0,4.0,0.00,2.0,0,1,182076.97,0,4.0,2,441.0,0.0
3,4.0,Boni,515.0,0,0,40.0,0.0,109542.29,1.0,1,1,166370.81,0,5.0,1,312.0,0.0
4,5.0,Jamieson,528.0,2,0,43.0,7.0,97473.87,2.0,1,1,159823.16,0,3.0,2,654.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8025,8026.0,Ma,490.0,1,0,41.0,0.0,139659.04,1.0,1,1,176254.12,0,5.0,3,312.0,0.0
8026,8027.0,Stevens,553.0,2,0,41.0,6.0,144974.55,1.0,1,1,19344.92,0,1.0,2,699.0,0.0
8027,8028.0,Defalco,808.0,0,0,41.0,0.0,0.00,1.0,1,1,79888.78,0,3.0,1,398.0,0.0
8028,8029.0,Docherty,706.0,0,1,34.5,5.0,0.00,1.0,0,0,164128.41,1,2.0,1,620.0,1.0


In [31]:
x = dftrain.drop(['ID', 'Surname','Exited'], axis=1)
y = dftrain['Exited']

In [32]:
# Membagi dataset menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [33]:
# Membuat model Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Melatih model
model.fit(X_train, y_train)

# Melakukan prediksi
y_pred = model.predict(X_test)

##Evaluasi

In [34]:
# Menghitung F1 Score
f1 = f1_score(y_test, y_pred, average='weighted')
print(f'F1 Score: {f1:.2f}')

F1 Score: 1.00


##Train To Data Test

In [35]:
# Mapping untuk kolom kategorikal
dftest['Gender'] = dftest['Gender'].map({'Male': 0, 'Female': 1})
dftest['HasCrCard'] = dftest['HasCrCard'].map({'No': 0, 'Yes': 1})
dftest['IsActiveMember'] = dftest['IsActiveMember'].map({'No': 0, 'Yes': 1})
dftest['Complain'] = dftest['Complain'].map({'No': 0, 'Yes': 1})

In [36]:
predicction = model.predict(dftest.drop(['ID', 'Surname'], axis=1))

In [37]:
predicction

array([1., 0., 0., ..., 0., 0., 0.])

In [39]:
df_predictions = pd.DataFrame({
    'ID': dftest['ID'],
    'Prediksi': predicction
})

In [40]:
# Menyimpan hasil prediksi ke file CSV
df_predictions.to_csv('predictions.csv', index=False)